# 🎨 Anime Mood Detector - Google Colab Training

Complete training pipeline for ResNet-50 emotion detector on FER2013 dataset.

**Duration:** ~2-3 hours on GPU
**Output:** Trained model saved to Google Drive

## 1️⃣ Setup & Installation

First, we'll install required packages and verify GPU availability.

In [ ]:
# Install required packages
!pip install --upgrade torch torchvision torchaudio
!pip install opencv-python mediapipe scikit-learn tqdm kaggle pandas numpy pillow matplotlib

# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ GPU not available - training will be slow")

## 2️⃣ Mount Google Drive

This allows us to save the trained model to your Drive.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
print("✓ Google Drive mounted at /content/drive")

# Create working directory
WORK_DIR = '/content/drive/MyDrive/anime-mood-detector'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print(f"Working directory: {WORK_DIR}")

## 3️⃣ Setup Kaggle API

You need Kaggle credentials to download the FER2013 dataset.

**Steps:**
1. Go to https://www.kaggle.com/settings/account
2. Click "Create New API Token"
3. Upload the `kaggle.json` file below
4. Or paste credentials manually

In [ ]:
from google.colab import files
import os
import json

# Setup Kaggle credentials
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)

# Option 1: Upload kaggle.json file
print("📁 OPTION 1: Upload kaggle.json file")
print("(Skip this if you want to paste API key instead)\n")

# Option 2: Paste API credentials directly
print("📝 OPTION 2: Paste API credentials directly")
print("Enter your Kaggle username:")
username = input().strip()

print("Enter your Kaggle API key (starts with KGAT_):")
api_key = input().strip()

if username and api_key:
    # Create kaggle.json
    kaggle_config = {
        "username": username,
        "key": api_key
    }
    
    kaggle_json_path = os.path.join(kaggle_dir, 'kaggle.json')
    with open(kaggle_json_path, 'w') as f:
        json.dump(kaggle_config, f)
    
    os.chmod(kaggle_json_path, 0o600)
    print("✓ Kaggle credentials configured successfully!")
else:
    print("⚠️ Credentials not provided. Skipping...")
    print("\nAlternatively, upload kaggle.json file below:")
    try:
        uploaded = files.upload()
        if 'kaggle.json' in uploaded:
            import shutil
            shutil.move('kaggle.json', os.path.join(kaggle_dir, 'kaggle.json'))
            os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600)
            print("✓ Kaggle credentials configured")
    except Exception as e:
        print(f"Note: {e}")

## 4️⃣ Download FER2013 Dataset

Download the FER2013 emotion dataset from Kaggle (~300 MB).

In [ ]:
import os
import subprocess
import zipfile
from pathlib import Path

# Download FER2013 dataset
data_dir = os.path.join(WORK_DIR, 'data')
os.makedirs(data_dir, exist_ok=True)

print("📥 Downloading FER2013 dataset from Kaggle...")
print("This may take 5-10 minutes...\n")

try:
    # Download fer2013 dataset
    result = subprocess.run(
        ['kaggle', 'datasets', 'download', '-d', 'msambare/fer2013', '-p', data_dir],
        capture_output=True,
        text=True
    )
    
    if result.returncode == 0:
        print("✓ Dataset downloaded")
    else:
        print(f"⚠️ Download issue: {result.stderr}")
        
except Exception as e:
    print(f"❌ Error: {e}")
    print("Make sure Kaggle credentials are properly configured")

# Extract ZIP file if it exists
zip_path = os.path.join(data_dir, 'fer2013.zip')
if os.path.exists(zip_path):
    print(f"\n📦 Extracting {zip_path}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(data_dir)
    print("✓ Extraction complete")
    # Clean up ZIP file
    os.remove(zip_path)
    print("✓ ZIP file cleaned up")

# Verify dataset structure
fer2013_dir = os.path.join(data_dir, 'fer2013')
if os.path.exists(fer2013_dir):
    print(f"\n✓ Dataset directory: {fer2013_dir}")
    subdirs = os.listdir(fer2013_dir)
    for subdir in sorted(subdirs):
        path = os.path.join(fer2013_dir, subdir)
        if os.path.isdir(path):
            count = len(os.listdir(path))
            print(f"  - {subdir}: {count} items")
else:
    print(f"Dataset not found at {fer2013_dir}")

In [ ]:
import os

# Count images in each folder
data_dir = '/content/drive/MyDrive/anime-mood-detector/data'
fer_dir = os.path.join(data_dir, 'fer2013')

print("📊 FER2013 Dataset Statistics:\n")

total_images = 0
for split in ['train', 'test']:
    split_dir = os.path.join(fer_dir, split)
    if os.path.exists(split_dir):
        print(f"📁 {split.upper()}:")
        split_total = 0
        for emotion in sorted(os.listdir(split_dir)):
            emotion_dir = os.path.join(split_dir, emotion)
            if os.path.isdir(emotion_dir):
                count = len([f for f in os.listdir(emotion_dir) if f.endswith(('.jpg', '.jpeg', '.png'))])
                print(f"  {emotion:12} {count:5} images")
                split_total += count
                total_images += count
        print(f"  {'TOTAL':12} {split_total:5} images\n")

print(f"✅ TOTAL IMAGES: {total_images}")

if total_images > 20000:
    print("✓ Dataset looks complete! Ready for training.")
else:
    print(f"⚠️ Dataset may be incomplete (expected ~35,000 images)")

## 5️⃣ Define Configuration

All code is self-contained in this notebook. No file uploads needed!

In [ ]:
import torch
import os

# Configuration
WORK_DIR = '/content/drive/MyDrive/anime-mood-detector'
DATA_DIR = os.path.join(WORK_DIR, 'data')
FER2013_DIR = os.path.join(DATA_DIR, 'fer2013')
MODELS_DIR = os.path.join(WORK_DIR, 'models')

os.makedirs(FER2013_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# Emotions
EMOTIONS = {
    0: 'angry',
    1: 'disgust',
    2: 'fear',
    3: 'happy',
    4: 'sad',
    5: 'surprise',
    6: 'neutral'
}

EMOTION_LABELS = list(EMOTIONS.values())
NUM_EMOTIONS = len(EMOTIONS)

# Model config
INPUT_SIZE = 224
NUM_CLASSES = NUM_EMOTIONS

# Training
STAGE1_EPOCHS = 15
STAGE1_BATCH_SIZE = 32
STAGE1_LR = 0.001

STAGE2_EPOCHS = 30
STAGE2_BATCH_SIZE = 32
STAGE2_LR = 0.0001

WEIGHT_DECAY = 5e-4  # Increased from 1e-4 to reduce overfitting
EARLY_STOPPING_PATIENCE = 5
RANDOM_SEED = 42

# Normalization
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
TRAINED_MODEL_PATH = os.path.join(MODELS_DIR, 'emotion_detector_final.pth')
BEST_MODEL_PATH = os.path.join(MODELS_DIR, 'emotion_detector_best.pth')

print("✓ Configuration loaded")
print(f"  Device: {DEVICE}")
print(f"  Work dir: {WORK_DIR}")
print(f"  Emotions: {NUM_EMOTIONS}")

In [ ]:
import os
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split

class FER2013Dataset(Dataset):
    """FER2013 emotion dataset."""
    
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        
        # Load image
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

def get_data_loaders(batch_size=32, num_workers=2):
    """Create train, val, test data loaders."""
    
    # Collect images and labels
    image_paths = []
    labels = []
    
    for emotion_idx, emotion in EMOTIONS.items():
        emotion_dir = os.path.join(FER2013_DIR, emotion)
        if not os.path.exists(emotion_dir):
            print(f"Warning: {emotion_dir} not found")
            continue
        
        for img_file in os.listdir(emotion_dir):
            if img_file.endswith(('.jpg', '.jpeg', '.png')):
                image_paths.append(os.path.join(emotion_dir, img_file))
                labels.append(emotion_idx)
    
    image_paths = np.array(image_paths)
    labels = np.array(labels)
    
    print(f"Total images: {len(image_paths)}")
    
    # Split: 70% train, 15% val, 15% test
    train_paths, temp_paths, train_labels, temp_labels = train_test_split(
        image_paths, labels, test_size=0.3, random_state=42, stratify=labels
    )
    val_paths, test_paths, val_labels, test_labels = train_test_split(
        temp_paths, temp_labels, test_size=0.5, random_state=42, stratify=temp_labels
    )
        # Augmentation (increased to reduce overfitting)
    train_transform = transforms.Compose([
        transforms.RandomRotation(20),  # Increased from 10
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),  # More aggressive
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),  # Random shift
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 0.2)),  # Added blur
        transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD)
    ])
    
    val_test_transform = transforms.Compose([
        transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD)
    ])
    
    # Datasets
    train_dataset = FER2013Dataset(train_paths, train_labels, train_transform)
    val_dataset = FER2013Dataset(val_paths, val_labels, val_test_transform)
    test_dataset = FER2013Dataset(test_paths, test_labels, val_test_transform)
    
    # Loaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    
    print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")
    
    return train_loader, val_loader, test_loader

print("✓ Dataset loader defined")

## 7️⃣ Define Model & Trainer

In [ ]:
import torch.nn as nn
from torchvision import models
from torch.optim import Adam, SGD
from torch.optim.lr_scheduler import ReduceLROnPlateau

class EmotionDetector(nn.Module):
    """ResNet-50 based emotion classifier."""
    
    def __init__(self, num_classes=7, pretrained=True):
        super().__init__()
        
        # Load pretrained ResNet-50
        weights = models.ResNet50_Weights.DEFAULT if pretrained else None
        self.resnet = models.resnet50(weights=weights)
        
        # Replace final FC layer with dropout + FC
        num_features = self.resnet.fc.in_features
        self.dropout = nn.Dropout(p=0.3)  # 30% dropout
        self.resnet.fc = nn.Linear(num_features, num_classes)
    
    def forward(self, x):
        x = self.resnet.conv1(x)
        x = self.resnet.bn1(x)
        x = self.resnet.relu(x)
        x = self.resnet.maxpool(x)
        x = self.resnet.layer1(x)
        x = self.resnet.layer2(x)
        x = self.resnet.layer3(x)
        x = self.resnet.layer4(x)
        x = self.resnet.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)  # Apply dropout before final layer
        x = self.resnet.fc(x)
        return x
    
    def freeze_backbone(self):
        """Freeze all layers except fc."""
        for param in self.resnet.layer1.parameters():
            param.requires_grad = False
        for param in self.resnet.layer2.parameters():
            param.requires_grad = False
        for param in self.resnet.layer3.parameters():
            param.requires_grad = False
        for param in self.resnet.layer4.parameters():
            param.requires_grad = False
    
    def unfreeze_backbone(self):
        """Unfreeze all layers."""
        for param in self.resnet.parameters():
            param.requires_grad = True

## 6️⃣ Prepare Data

Load the FER2013 dataset and create data loaders.

In [ ]:
# Load data
print("📊 Loading FER2013 dataset...")

train_loader, val_loader, test_loader = get_data_loaders(
    batch_size=STAGE1_BATCH_SIZE,
    num_workers=2
)

print("✓ Data loaders created")

## 7️⃣ Train Model

Two-stage training:
- **Stage 1** (15 epochs): Frozen backbone, train classification head only
- **Stage 2** (30 epochs): Fine-tune entire network

**Estimated time:** ~2-3 hours on GPU

In [ ]:
import time

# Create model
print("🏗️ Creating ResNet-50 model...")
model = EmotionDetector(num_classes=NUM_EMOTIONS, pretrained=True)
model = model.to(DEVICE)

print(f"Model on device: {DEVICE}")
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

# Create trainer
trainer = EmotionTrainer(model, DEVICE)

# Train
print("\n🚀 Starting two-stage training...")
start_time = time.time()

trainer.train_two_stage(train_loader, val_loader)

elapsed_time = time.time() - start_time
hours = elapsed_time / 3600
print(f"\n✓ Training complete in {hours:.1f} hours ({elapsed_time/60:.1f} minutes)")

## 8️⃣ Save Training Checkpoint

Save current model state to resume training later without losing progress.


In [ ]:
import json

# Save checkpoint with full training state
CHECKPOINT_PATH = os.path.join(MODELS_DIR, 'training_checkpoint.pth')
CHECKPOINT_META_PATH = os.path.join(MODELS_DIR, 'checkpoint_metadata.json')

# Save model + trainer state
checkpoint = {
    'model_state_dict': model.state_dict(),
    'best_val_acc': trainer.best_val_acc,
    'patience_counter': trainer.patience_counter,
}

torch.save(checkpoint, CHECKPOINT_PATH)

# Save metadata
metadata = {
    'best_model_path': BEST_MODEL_PATH,
    'checkpoint_path': CHECKPOINT_PATH,
    'best_val_acc': float(trainer.best_val_acc),
    'note': 'Load this checkpoint to resume training from current state'
}

with open(CHECKPOINT_META_PATH, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"✓ Training checkpoint saved!")
print(f"  Model checkpoint: {CHECKPOINT_PATH}")
print(f"  Metadata: {CHECKPOINT_META_PATH}")
print(f"  Best validation accuracy: {trainer.best_val_acc:.2f}%")
print(f"\n💾 You can now resume training from this point without losing progress!")


### 📂 Resume from Checkpoint (Run this cell INSTEAD of "Train Model" if resuming)

Skip this cell on first run. Use it only when resuming training from a saved checkpoint.


In [ ]:
import time
import json

# Load checkpoint (only run this cell when RESUMING training)
CHECKPOINT_PATH = os.path.join(MODELS_DIR, 'training_checkpoint.pth')
CHECKPOINT_META_PATH = os.path.join(MODELS_DIR, 'checkpoint_metadata.json')

if os.path.exists(CHECKPOINT_PATH):
    print("📂 Loading from training checkpoint...")
    
    # Load checkpoint
    checkpoint = torch.load(CHECKPOINT_PATH)
    
    # Recreate model and load state
    print("🏗️ Recreating ResNet-50 model...")
    model = EmotionDetector(num_classes=NUM_EMOTIONS, pretrained=True)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(DEVICE)
    
    # Recreate trainer and restore state
    trainer = EmotionTrainer(model, DEVICE)
    trainer.best_val_acc = checkpoint['best_val_acc']
    trainer.patience_counter = checkpoint['patience_counter']
    
    # Load metadata
    with open(CHECKPOINT_META_PATH, 'r') as f:
        metadata = json.load(f)
    
    print(f"✓ Checkpoint loaded!")
    print(f"  Previous best accuracy: {trainer.best_val_acc:.2f}%")
    print(f"  Patience counter: {trainer.patience_counter}")
    
    # Ready to resume training
    print(f"\n🚀 Ready to resume training from saved state!")
    print(f"Run trainer.train_two_stage(train_loader, val_loader) to continue")
    
else:
    print("❌ Checkpoint not found!")
    print(f"Expected at: {CHECKPOINT_PATH}")
    print("Make sure you've run the 'Save Training Checkpoint' cell first")


## 8️⃣ Evaluate Model

Test the trained model on validation and test sets.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Load best model
print("📊 Evaluating model...")
model.load_state_dict(torch.load(BEST_MODEL_PATH))
model.eval()

criterion = nn.CrossEntropyLoss()

# Validation
val_loss = 0
val_correct = 0
val_total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        val_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        val_correct += (predicted == labels).sum().item()
        val_total += labels.size(0)

val_accuracy = 100 * val_correct / val_total
print(f"✓ Validation - Loss: {val_loss/len(val_loader):.4f}, Accuracy: {val_accuracy:.2f}%")

# Test
test_loss = 0
test_correct = 0
test_total = 0
all_predicted = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        test_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        test_correct += (predicted == labels).sum().item()
        test_total += labels.size(0)
        
        all_predicted.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_accuracy = 100 * test_correct / test_total
print(f"✓ Test - Loss: {test_loss/len(test_loader):.4f}, Accuracy: {test_accuracy:.2f}%")

# Classification report
print("\n📈 Classification Report:")
print(classification_report(all_labels, all_predicted, target_names=EMOTION_LABELS))

## 9️⃣ Save Model to Google Drive

The model is automatically saved to Google Drive. Check your Drive at:
`/My Drive/anime-mood-detector/models/`

In [ ]:
import os

# Check saved models
print("📁 Saved models:")
if os.path.exists(BEST_MODEL_PATH):
    size = os.path.getsize(BEST_MODEL_PATH) / 1e6
    print(f"  ✓ Best model: {BEST_MODEL_PATH} ({size:.1f} MB)")

if os.path.exists(TRAINED_MODEL_PATH):
    size = os.path.getsize(TRAINED_MODEL_PATH) / 1e6
    print(f"  ✓ Final model: {TRAINED_MODEL_PATH} ({size:.1f} MB)")

print(f"\n✓ Models saved to Google Drive at:")
print(f"  /My Drive/anime-mood-detector/models/")

print("\n✅ TRAINING COMPLETE!")
print("\n📝 Next steps:")
print("1. Download both .pth files from Google Drive")
print("2. Place them in: c:\\Users\\xx\\Desktop\\alya\\models\\")
print("3. Run: streamlit run src/app.py")
print("\n🎉 Your Anime Mood Detector is ready!")

## 🔟 Download Models Manually

Download trained models directly to your machine.


In [ ]:
from google.colab import files

# Define paths (in case they're not in scope)
CHECKPOINT_PATH = os.path.join(MODELS_DIR, 'training_checkpoint.pth')

# Download models
print("📥 Downloading trained models...\n")

files_to_download = [
    (BEST_MODEL_PATH, 'emotion_detector_best.pth'),
    (TRAINED_MODEL_PATH, 'emotion_detector_final.pth'),
    (CHECKPOINT_PATH, 'training_checkpoint.pth'),
]

for src_path, filename in files_to_download:
    if os.path.exists(src_path):
        print(f"⬇️  Downloading {filename}...")
        files.download(src_path)
    else:
        print(f"⚠️  {filename} not found at {src_path}")

print("\n✅ Downloads complete!")
print("\n📁 Files downloaded:")
print("  • emotion_detector_best.pth")
print("  • emotion_detector_final.pth")
print("  • training_checkpoint.pth")
print("\n📝 Next: Move these files to c:\\Users\\xx\\Desktop\\alya\\models\\")


## Summary

You've successfully trained an emotion detector! 🎉

### 📊 Dataset
- **FER2013**: 35,887 total images
- **7 Emotions**: angry, disgust, fear, happy, sad, surprise, neutral
- **Image Size**: 48×48 pixels (grayscale)
- **Split**: 70% train, 15% val, 15% test

### 🧠 Model
- **Architecture**: ResNet-50 (25.5M parameters)
- **Pre-training**: ImageNet
- **Two-stage fine-tuning**:
  - Stage 1: Frozen backbone (15 epochs)
  - Stage 2: Full network (30 epochs)

### 📈 Expected Performance
- **Validation Accuracy**: 65-70%
- **Test Accuracy**: Similar to validation
- **Training Time**: ~2-3 hours on GPU
- **Model Size**: ~120 MB

### 🎨 Integration with Your App
1. Download the trained models from Google Drive
2. Place in `models/` directory
3. Run: `streamlit run src/app.py`
4. Your Alya character will detect emotions in real-time!

### 🚀 Next Steps
- Download models: `emotion_detector_best.pth` and `emotion_detector_final.pth`
- Save to local: `c:\Users\xx\Desktop\alya\models\`
- Test with webcam or images
- Fine-tune further if needed